# Reusable Fracture Detection Trainer (any body part)

**Learning project, not a medical device.**

**How to use for a new body part:** change ONLY the settings in cell 1 (`REGION`, data source, folder), then **Runtime > Run all**. At the end you get `model_<REGION>.pt` and `results_<REGION>.json`.

Works on **Google Colab** and **Kaggle Notebooks** (it detects which one it is on).

- Colab: **Runtime > Change runtime type > T4 GPU**.
- Kaggle: right panel **Session options > Accelerator > GPU**, and turn **Internet on**.

Your data needs two kinds of folders, named so the notebook can tell them apart, for example `fractured/` and `not fractured/` (or `fracture/` and `normal/`). It may also already be split into `train/`, `val/`, `test/` folders. If there is no split, the notebook makes one itself (70/15/15).

## 1. Settings (the only cell you edit)

In [ ]:
# ---------- EDIT THESE ----------
REGION = "multi_region_demo"      # name used in saved files, e.g. "ankle", "wrist", "knee"

SOURCE = "kaggle"                 # "kaggle" | "input" | "drive" | "zip" | "path"
                                  # "input" = a dataset you added with Add Input on Kaggle (auto-found in /kaggle/input)
KAGGLE_ID = "bmadushanirodrigo/fracture-multi-region-x-ray-data"   # used when SOURCE = "kaggle"
DATA_PATH = ""                    # folder (SOURCE "drive"/"path") or .zip file (SOURCE "zip")
                                  # e.g. "/content/drive/MyDrive/ankle_data"

# Optional: set BOTH if automatic folder-name detection guesses wrong.
POS_FOLDER = None                 # exact name of the fracture folder, e.g. "fracture"
NEG_FOLDER = None                 # exact name of the normal folder, e.g. "normal"

# ---------- usually leave these ----------
EPOCHS = 15
BATCH_SIZE = 32
LR = 1e-4
IMG_SIZE = 224
TARGET_SENSITIVITY = 0.95         # threshold is picked on validation data to reach this
DEDUP_AND_RESPLIT = True          # recommended: pool all images, keep one copy of each identical image, then split 70/15/15
                                  # (ignores the dataset's own train/val/test folders, which may share identical images)
REMOVE_TEST_DUPLICATES = True     # also drop val/test images that are byte-identical to a training image
SAVE_TO_DRIVE = False             # Colab only: True = also copy results into Google Drive
SEED = 42

# DIP experiment: image preprocessing applied to EVERY image (train, val, test and the app)
PREPROCESS = "none"               # "none" | "clahe" | "denoise" | "clahe_denoise"
                                  # tip: change REGION too, e.g. "multi_region_demo_clahe", so files do not overwrite each other

# EXTERNAL TEST (optional): a SECOND dataset used ONLY for testing, never for training
EXTERNAL_SOURCE = "none"          # "none" (skip) | "kaggle" | "input" | "drive" | "zip" | "path"
EXTERNAL_KAGGLE_ID = ""           # used when EXTERNAL_SOURCE = "kaggle", e.g. "owner/dataset-name"
EXTERNAL_PATH = ""                # folder (or .zip) for "input"/"drive"/"zip"/"path", e.g. "/kaggle/input/my-second-dataset"
EXTERNAL_POS_FOLDER = None        # set BOTH if folder names are not obvious, e.g. "fracture"
COMBINE_EXTERNAL = False          # True = ALSO TRAIN on the second dataset (pooled). Cell 10c then reports the test score per dataset.
                                  # tip: change REGION too, e.g. "multi_region_combined"
EXTERNAL_NEG_FOLDER = None        #                                              e.g. "normal"

## 2. Setup

In [ ]:
import os, sys, io, json, random, hashlib, shutil, zipfile, warnings, subprocess
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from sklearn.metrics import roc_auc_score, confusion_matrix, roc_curve
from sklearn.model_selection import GroupShuffleSplit
import matplotlib.pyplot as plt
from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True      # tolerate slightly damaged files
try:
    import cv2
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'opencv-python-headless']); import cv2

def dip_preprocess(img, mode, size):
    # Classical DIP preprocessing. The app uses the same function. mode: none | clahe | denoise | clahe_denoise
    if mode == 'none': return img
    g = np.array(img.convert('L').resize((size, size), Image.BILINEAR))
    if 'denoise' in mode: g = cv2.medianBlur(g, 3)                                  # 3x3 median filter (removes speckle noise)
    if 'clahe' in mode: g = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(g)   # local contrast enhancement
    return Image.fromarray(g).convert('RGB')
warnings.filterwarnings('ignore', category=UserWarning)

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
IN_KAGGLE = os.path.exists('/kaggle/working')
IN_COLAB = (not IN_KAGGLE) and ('google.colab' in sys.modules or 'COLAB_GPU' in os.environ)
WORK_DIR = '/kaggle/working' if IN_KAGGLE else ('/content' if IN_COLAB else '.')
print('Running on:', 'Kaggle' if IN_KAGGLE else 'Colab' if IN_COLAB else 'other computer')
print('Region:', REGION, '| Device:', device)
if device.type == 'cpu':
    print('WARNING: no GPU. Colab: Runtime > Change runtime type > T4 GPU. Kaggle: Session options > Accelerator > GPU.')

## 3. Get the data

In [ ]:
if SOURCE == 'kaggle':
    subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'kagglehub'])
    import kagglehub
    DATA_ROOT = kagglehub.dataset_download(KAGGLE_ID)
elif SOURCE == 'input':                                   # Kaggle: dataset added via "Add Input"
    DATA_ROOT = DATA_PATH or '/kaggle/input'
    print('Folders in /kaggle/input:', os.listdir('/kaggle/input') if os.path.isdir('/kaggle/input') else 'none')
elif SOURCE == 'drive':
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_ROOT = DATA_PATH
elif SOURCE == 'zip':
    DATA_ROOT = os.path.join(WORK_DIR, 'data_' + REGION)
    os.makedirs(DATA_ROOT, exist_ok=True)
    with zipfile.ZipFile(DATA_PATH) as z:
        z.extractall(DATA_ROOT)
else:
    DATA_ROOT = DATA_PATH

assert os.path.isdir(DATA_ROOT), f'Data folder not found: {DATA_ROOT}'
print('Data folder:', DATA_ROOT)

def get_root(source, kaggle_id, path, tag):
    if source == 'kaggle':
        subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'kagglehub']); import kagglehub
        return kagglehub.dataset_download(kaggle_id)
    assert path, 'Set EXTERNAL_PATH to the exact folder (or zip) of the second dataset'
    if source == 'drive':
        from google.colab import drive; drive.mount('/content/drive'); return path
    if source == 'zip':
        out = os.path.join(WORK_DIR, 'data_' + tag); os.makedirs(out, exist_ok=True)
        with zipfile.ZipFile(path) as z: z.extractall(out)
        return out
    return path
for root, dirs, files in os.walk(DATA_ROOT):
    depth = root[len(DATA_ROOT):].count(os.sep)
    if depth <= 2:
        print('  ' * depth + os.path.basename(root) + f'/  ({len(files)} files)')

## 4. Find images, labels and splits

Label rule: the nearest folder name decides it. Names like *not fractured, normal, negative, healthy, no fracture* = **normal (0)**; names like *fractured, fracture, positive, abnormal* = **fracture (1)**. Set `POS_FOLDER` / `NEG_FOLDER` in cell 1 to override.

In [ ]:
IMG_EXT = ('.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff')
SPLIT_NAMES = {'train': 'train', 'training': 'train', 'val': 'val', 'valid': 'val',
               'validation': 'val', 'test': 'test', 'testing': 'test'}
NEG_WORDS = ['not', 'normal', 'non', 'negative', 'healthy', 'nofrac', 'no_frac', 'no frac', 'no-frac', 'intact']
POS_WORDS = ['frac', 'positive', 'broken', 'abnormal', 'injur']

def folder_label(name):
    n = name.lower()
    if POS_FOLDER and NEG_FOLDER:                       # explicit mode
        if n == POS_FOLDER.lower(): return 1
        if n == NEG_FOLDER.lower(): return 0
        return None
    if 'abnormal' in n: return 1                        # 'abnormal' contains 'normal'
    if any(w in n for w in NEG_WORDS): return 0
    if any(w in n for w in POS_WORDS): return 1
    return None

def scan(root):
    items = []                                          # (path, label, split, group)
    for dp, _, fs in os.walk(root):
        for fn in fs:
            if not fn.lower().endswith(IMG_EXT): continue
            p = os.path.join(dp, fn)
            parts = os.path.relpath(p, root).split(os.sep)[:-1]   # folders only, inside root
            label, li = None, None
            for i in range(len(parts) - 1, -1, -1):
                l = folder_label(parts[i])
                if l is not None: label, li = l, i; break
            if label is None: continue
            split = next((SPLIT_NAMES[x.lower()] for x in parts if x.lower() in SPLIT_NAMES), None)
            group = '/'.join(parts[:li + 2]) if len(parts) > li + 1 else p   # keeps patient/study folders together
            items.append((p, label, split, group))
    return items

def group_split(items, frac, seed):
    gss = GroupShuffleSplit(n_splits=1, test_size=frac, random_state=seed)
    a, b = next(gss.split(np.arange(len(items)), groups=[it[3] for it in items]))
    return [items[i] for i in a], [items[i] for i in b]

items = scan(DATA_ROOT)
assert len(items) > 0, 'No labelled images found. Check folder names or set POS_FOLDER / NEG_FOLDER.'
pre = {k: [it for it in items if it[2] == k] for k in ('train', 'val', 'test')}
items = [it + ('A',) for it in items]                    # 5th field = which dataset the image came from
if COMBINE_EXTERNAL:
    assert EXTERNAL_SOURCE != 'none', 'COMBINE_EXTERNAL needs EXTERNAL_SOURCE and EXTERNAL_PATH / EXTERNAL_KAGGLE_ID'
    _ext_root = get_root(EXTERNAL_SOURCE, EXTERNAL_KAGGLE_ID, EXTERNAL_PATH, 'external')
    assert os.path.isdir(_ext_root), f'External folder not found: {_ext_root}'
    _p, _n = POS_FOLDER, NEG_FOLDER
    POS_FOLDER, NEG_FOLDER = EXTERNAL_POS_FOLDER, EXTERNAL_NEG_FOLDER
    _b = scan(_ext_root)
    POS_FOLDER, NEG_FOLDER = _p, _n
    assert len(_b) > 0, 'No labelled images found in the second dataset. Set EXTERNAL_POS_FOLDER / EXTERNAL_NEG_FOLDER.'
    items += [(p, l, sp, 'B:' + g, 'B') for (p, l, sp, g) in _b]
    print(f'Dataset A: {sum(1 for it in items if it[4] == "A")} images | Dataset B: {len(_b)} images (pooled for training)')

def md5(p):
    h = hashlib.md5()
    with open(p, 'rb') as f:
        for c in iter(lambda: f.read(1 << 20), b''): h.update(c)
    return h.hexdigest()

if DEDUP_AND_RESPLIT or COMBINE_EXTERNAL:
    by_hash = {}
    for it in items: by_hash.setdefault(md5(it[0]), []).append(it)
    uniq, conflicts = [], 0
    for lst in by_hash.values():
        if len({x[1] for x in lst}) > 1: conflicts += 1; continue     # same image under both labels: drop it
        uniq.append(lst[0])
    print(f'{len(items)} images found -> {len(uniq)} unique ({len(items) - len(uniq) - conflicts} identical copies removed, {conflicts} images with conflicting labels dropped)')
    items = uniq
    print('Fresh 70/15/15 split of the unique images.' + (' (done separately inside each dataset, so both appear in train, val and test)' if COMBINE_EXTERNAL else ''))
    if COMBINE_EXTERNAL:
        train_items, val_items, test_items = [], [], []
        for _s in ('A', 'B'):
            sub = [it for it in items if it[4] == _s]
            rest, t = group_split(sub, 0.15, SEED); a, v = group_split(rest, 0.15 / 0.85, SEED)
            train_items += a; val_items += v; test_items += t
    else:
        rest, test_items = group_split(items, 0.15, SEED)
        train_items, val_items = group_split(rest, 0.15 / 0.85, SEED)
elif pre['train'] and pre['test']:
    print('Using the dataset\'s own train/val/test folders.')
    train_items, test_items = pre['train'], pre['test']
    if pre['val']: val_items = pre['val']
    else: train_items, val_items = group_split(train_items, 0.10, SEED)
else:
    print('No train/test folders found, making a 70/15/15 split (images from the same patient/study folder stay together).')
    rest, test_items = group_split(items, 0.15, SEED)
    train_items, val_items = group_split(rest, 0.15 / 0.85, SEED)

def summary(name, its):
    n1 = sum(it[1] for it in its); print(f'{name:6s} total {len(its):6d} | fractured {n1:6d} | normal {len(its) - n1:6d}')
for n, s in [('train', train_items), ('val', val_items), ('test', test_items)]: summary(n, s)
assert all(len({it[1] for it in s}) == 2 for s in (train_items, val_items, test_items)), 'A split is missing one class. Check folder names.'

## 5. Duplicate check (data leakage)

If the same image appears in train and test, the score looks better than it really is. This removes exact byte-for-byte copies from val/test. (Rotated or edited copies are not caught.)

In [ ]:
def md5(p):
    h = hashlib.md5()
    with open(p, 'rb') as f:
        for c in iter(lambda: f.read(1 << 20), b''): h.update(c)
    return h.hexdigest()

train_hashes = {md5(it[0]) for it in train_items}
def drop_dups(name, its):
    keep = [it for it in its if md5(it[0]) not in train_hashes]
    print(f'{name}: {len(its) - len(keep)} exact duplicates of training images' +
          (' (removed)' if REMOVE_TEST_DUPLICATES else ' (kept)'))
    return keep if REMOVE_TEST_DUPLICATES else its
val_items, test_items = drop_dups('val', val_items), drop_dups('test', test_items)
DUPES_REMOVED = REMOVE_TEST_DUPLICATES
for n, s in [('train', train_items), ('val', val_items), ('test', test_items)]: summary(n, s)
assert all(len({it[1] for it in s}) == 2 for s in (train_items, val_items, test_items)), 'After duplicate removal a split has only one class. Set REMOVE_TEST_DUPLICATES = False or check the data.'

## 6. Load data

In [ ]:
mean, std = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)), transforms.RandomHorizontalFlip(), transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2), transforms.Grayscale(3),
    transforms.ToTensor(), transforms.Normalize(mean, std)])
eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)), transforms.Grayscale(3),
    transforms.ToTensor(), transforms.Normalize(mean, std)])

class XrayDS(Dataset):
    def __init__(self, its, tf): self.items, self.tf = its, tf
    def __len__(self): return len(self.items)
    def __getitem__(self, i):
        for k in range(len(self.items)):                 # skip unreadable files
            p, l = self.items[(i + k) % len(self.items)][:2]
            try: return self.tf(dip_preprocess(Image.open(p).convert('RGB'), PREPROCESS, IMG_SIZE)), l
            except Exception: continue
        raise RuntimeError('No readable images')

train_ds, val_ds, test_ds = XrayDS(train_items, train_tf), XrayDS(val_items, eval_tf), XrayDS(test_items, eval_tf)
FRAC = 1                                                  # label 1 = fracture, always
train_dl = DataLoader(train_ds, BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_dl   = DataLoader(val_ds,   BATCH_SIZE, shuffle=False, num_workers=2)
test_dl  = DataLoader(test_ds,  BATCH_SIZE, shuffle=False, num_workers=2)

inv = lambda t: (t.permute(1, 2, 0) * torch.tensor(std) + torch.tensor(mean)).clamp(0, 1)
fig, axs = plt.subplots(2, 6, figsize=(16, 6))
for ax, i in zip(axs.ravel(), random.sample(range(len(train_ds)), 12)):
    x, y = train_ds[i]; ax.imshow(inv(x)); ax.set_title('fractured' if y == 1 else 'normal'); ax.axis('off')
plt.tight_layout(); plt.show()

## 6b. DIP demo (for your report, not used in training)

Shows classical image-processing operations on 3 test X-rays: histogram equalisation, CLAHE, median filter, Sobel edges, Otsu threshold and the Fourier spectrum, plus histograms. Run it once and screenshot it.

In [ ]:
def to_gray(path, size=IMG_SIZE):
    return np.array(Image.open(path).convert('L').resize((size, size), Image.BILINEAR))

def sobel(g):
    gx, gy = cv2.Sobel(g, cv2.CV_32F, 1, 0, ksize=3), cv2.Sobel(g, cv2.CV_32F, 0, 1, ksize=3)
    m = np.sqrt(gx ** 2 + gy ** 2); return (255 * m / (m.max() + 1e-8)).astype('uint8')

def fft_spectrum(g):
    f = np.fft.fftshift(np.fft.fft2(g)); m = np.log1p(np.abs(f)); return (255 * m / m.max()).astype('uint8')

picks = random.sample(test_items, 3)
cols = ['Original', 'Histogram equalisation', 'CLAHE', 'Median filter 3x3', 'Sobel edges', 'Otsu threshold', 'FFT spectrum (log)']
fig, axs = plt.subplots(3, len(cols), figsize=(21, 9))
for r, it in enumerate(picks):
    g = to_gray(it[0])
    outs = [g, cv2.equalizeHist(g), cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(g), cv2.medianBlur(g, 3),
            sobel(g), cv2.threshold(g, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)[1], fft_spectrum(g)]
    for c, o in enumerate(outs):
        axs[r, c].imshow(o, cmap='gray'); axs[r, c].axis('off')
        if r == 0: axs[r, c].set_title(cols[c])
plt.tight_layout(); plt.show()

g = to_gray(picks[0][0]); c = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(g)
plt.figure(figsize=(10, 3))
plt.subplot(1, 2, 1); plt.hist(g.ravel(), 256, range=(0, 255)); plt.title('Histogram: original')
plt.subplot(1, 2, 2); plt.hist(c.ravel(), 256, range=(0, 255)); plt.title('Histogram: after CLAHE')
plt.tight_layout(); plt.show()

## 7. Model (DenseNet121, pretrained)

In [ ]:
model = models.densenet121(weights=models.DenseNet121_Weights.DEFAULT)
model.classifier = nn.Linear(model.classifier.in_features, 2)
model = model.to(device)
counts = np.bincount([it[1] for it in train_items], minlength=2)
class_w = torch.tensor(counts.sum() / (2.0 * counts), dtype=torch.float).to(device)
criterion = nn.CrossEntropyLoss(weight=class_w)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

## 8. Train

In [ ]:
def evaluate(dl):
    model.eval(); probs, labels, loss_sum = [], [], 0.0
    with torch.no_grad():
        for x, y in dl:
            x, y = x.to(device), y.to(device)
            out = model(x)
            loss_sum += criterion(out, y).item() * len(y)
            probs += F.softmax(out, 1)[:, FRAC].cpu().tolist()
            labels += y.cpu().tolist()
    return loss_sum / len(dl.dataset), np.array(probs), np.array(labels)

MODEL_FILE = f'model_{REGION}.pt'
best_score, history = -1e9, []
for ep in range(EPOCHS):
    model.train(); run = 0.0
    for x, y in train_dl:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        loss = criterion(model(x), y); loss.backward(); optimizer.step()
        run += loss.item() * len(y)
    scheduler.step()
    vloss, vp, vl = evaluate(val_dl)
    vauc = roc_auc_score(vl, vp) if len(set(vl)) == 2 else float('nan')
    history.append((run / len(train_ds), vloss, vauc))
    print(f'Epoch {ep + 1:02d}/{EPOCHS}  train_loss {history[-1][0]:.4f}  val_loss {vloss:.4f}  val_AUC {vauc:.4f}')
    score = vauc if not np.isnan(vauc) else -vloss          # fall back to val loss if AUC is undefined
    if score > best_score:
        best_score = score; torch.save(model.state_dict(), MODEL_FILE)
print('Best score:', round(best_score, 4), '(val AUC, or minus val loss if AUC was undefined)')
if not os.path.exists(MODEL_FILE): raise RuntimeError('Model file was not saved')
h = np.array(history); plt.plot(h[:, 0], label='train loss'); plt.plot(h[:, 1], label='val loss')
plt.xlabel('epoch'); plt.legend(); plt.title('Loss'); plt.show()

## 9. Test (the real score)

The decision threshold is chosen on the **validation** set (to reach the target sensitivity), then the **test** set is scored once.

In [ ]:
model.load_state_dict(torch.load(MODEL_FILE, map_location=device))
_, vp, vl = evaluate(val_dl)
fpr, tpr, thr = roc_curve(vl, vp)
THRESH = float(thr[np.argmax(tpr >= TARGET_SENSITIVITY)])
_, tp, tl = evaluate(test_dl)
pred = (tp >= THRESH).astype(int)
tn, fp, fn, tpos = [int(v) for v in confusion_matrix(tl, pred, labels=[0, 1]).ravel()]
RESULTS = {
    'region': REGION, 'threshold': round(THRESH, 4), 'img_size': IMG_SIZE, 'mean': mean, 'std': std,
    'auc': round(float(roc_auc_score(tl, tp)), 4),
    'sensitivity': round(tpos / max(tpos + fn, 1), 4), 'specificity': round(tn / max(tn + fp, 1), 4),
    'accuracy': round((tpos + tn) / len(tl), 4),
    'confusion': {'TN': tn, 'FP': fp, 'FN': fn, 'TP': tpos},
    'n_train': len(train_items), 'n_val': len(val_items), 'n_test': len(test_items),
    'duplicates_removed_from_val_test': DUPES_REMOVED, 'label_1': 'fracture', 'preprocess': PREPROCESS,
    'note': 'Learning project, not a medical device.'}
print('TEST RESULTS for', REGION)
for k in ('threshold', 'auc', 'sensitivity', 'specificity', 'accuracy', 'confusion'): print(f'{k:12s}', RESULTS[k])
f, t, _ = roc_curve(tl, tp)
plt.plot(f, t); plt.plot([0, 1], [0, 1], '--'); plt.xlabel('False positive rate'); plt.ylabel('Sensitivity')
plt.title(f'ROC curve (test) - {REGION}'); plt.show()

## 10. Grad-CAM (is it looking at the bone?)

Check that the red areas sit on bone, not on corners or L/R markers. If they don't, do not trust the model.

In [ ]:
class GradCAM:
    def __init__(self, model, layer):
        self.model, self.acts, self.grads = model, None, None
        layer.register_forward_hook(lambda m, i, o: setattr(self, 'acts', o))
        layer.register_full_backward_hook(lambda m, gi, go: setattr(self, 'grads', go[0]))
    def __call__(self, x, cls):
        self.model.eval(); self.model.zero_grad()
        out = self.model(x); out[0, cls].backward()
        w = self.grads.mean(dim=(2, 3), keepdim=True)
        cam = F.relu((w * self.acts).sum(1)).squeeze().detach().cpu().numpy()
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam, F.softmax(out, 1)[0, cls].item()
cam_tool = GradCAM(model, model.features.denseblock4)

def show_cam(img, title_extra=''):
    x = img.unsqueeze(0).to(device).requires_grad_(True)
    cam, p = cam_tool(x, FRAC)
    cam = np.array(Image.fromarray((cam * 255).astype('uint8')).resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)) / 255.0
    plt.imshow(inv(img.cpu())); plt.imshow(cam, cmap='jet', alpha=0.4); plt.axis('off')
    plt.title(f'P(fracture)={p:.2f} {title_extra}', fontsize=10)
    return p

pos = [it for it in test_items if it[1] == 1]; neg = [it for it in test_items if it[1] == 0]
sel = random.sample(pos, min(5, len(pos))) + random.sample(neg, min(5, len(neg)))
plt.figure(figsize=(20, 9))
for k, it in enumerate(sel):
    plt.subplot(2, 5, k + 1); show_cam(eval_tf(dip_preprocess(Image.open(it[0]).convert('RGB'), PREPROCESS, IMG_SIZE)), '| true: ' + ('fractured' if it[1] else 'normal'))
plt.tight_layout(); plt.show()

## 10b. Unsupported-image check (saves `ood_<REGION>.npz`)

Saves a summary of what the training images look like to the model. The app compares each uploaded image with it and says **"unsupported image"** instead of guessing when an image looks too different (for example another body part or a watermarked stock photo). The limit is set on the validation images, so about 1% of normal images may be flagged.

In [ ]:
from sklearn.covariance import LedoitWolf
model.load_state_dict(torch.load(MODEL_FILE, map_location=device)); model.eval()

def embed(dl):
    out = []
    with torch.no_grad():
        for x, _ in dl:
            f = F.adaptive_avg_pool2d(F.relu(model.features(x.to(device))), 1).flatten(1)
            out.append(f.cpu().numpy())
    return np.concatenate(out)

train_eval_dl = DataLoader(XrayDS(train_items, eval_tf), BATCH_SIZE, shuffle=False, num_workers=2)
Ftr, Fva, Fte = embed(train_eval_dl), embed(val_dl), embed(test_dl)
lw = LedoitWolf().fit(Ftr)
MU, PREC = lw.location_, lw.precision_
def maha(Fx):
    d = Fx - MU
    return np.sqrt(np.maximum(np.einsum('ij,jk,ik->i', d, PREC, d), 0))
dva, dte = maha(Fva), maha(Fte)
OOD_THRESH = float(np.percentile(dva, 99))
OOD_FILE = f'ood_{REGION}.npz'
np.savez(OOD_FILE, mu=MU.astype('float32'), prec=PREC.astype('float32'), thresh=np.float32(OOD_THRESH))
print(f'Distance limit (99th percentile of validation images): {OOD_THRESH:.2f}')
print(f'Median distance  val {np.median(dva):.2f} | test {np.median(dte):.2f}')
print(f'Test images flagged as unsupported: {(dte > OOD_THRESH).mean():.1%} (should be small, around 1-3%)')
print('Saved:', OOD_FILE)

## 10c. External test (second dataset, never used for training)

If `EXTERNAL_SOURCE` is set in cell 1, the trained model is tested on a completely different dataset. This shows how well it works on new data. Exact copies of training/validation images are removed first. A big drop compared with the internal test usually means the new X-rays look different (different hospital, machine or body part).

In [ ]:
RESULTS['external'] = None
if COMBINE_EXTERNAL:
    model.load_state_dict(torch.load(MODEL_FILE, map_location=device))
    RESULTS['per_dataset'] = {}
    print('PER-DATASET TEST RESULTS (both datasets were used for training; test images are held out)')
    print(f"{'dataset':10s}{'n':>7s}{'AUC':>9s}{'sens':>9s}{'spec':>9s}   confusion")
    for _s, _nm in (('A', 'dataset 1'), ('B', 'dataset 2')):
        sub = [it for it in test_items if it[4] == _s]
        _, sp_, sl_ = evaluate(DataLoader(XrayDS(sub, eval_tf), BATCH_SIZE, shuffle=False, num_workers=2))
        pr_ = (sp_ >= THRESH).astype(int)
        a_, b_, c_, d_ = [int(v) for v in confusion_matrix(sl_, pr_, labels=[0, 1]).ravel()]
        r_ = {'n': len(sub), 'auc': round(float(roc_auc_score(sl_, sp_)), 4), 'sensitivity': round(d_ / max(d_ + c_, 1), 4),
              'specificity': round(a_ / max(a_ + b_, 1), 4), 'confusion': {'TN': a_, 'FP': b_, 'FN': c_, 'TP': d_}}
        RESULTS['per_dataset'][_nm] = r_
        print(f"{_nm:10s}{r_['n']:7d}{r_['auc']:9.4f}{r_['sensitivity']:9.4f}{r_['specificity']:9.4f}   {r_['confusion']}")
    RESULTS['combined'] = True
elif EXTERNAL_SOURCE == 'none':
    print('External test skipped (EXTERNAL_SOURCE = "none").')
else:
    def get_root(source, kaggle_id, path, tag):
        if source == 'kaggle':
            subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'kagglehub']); import kagglehub
            return kagglehub.dataset_download(kaggle_id)
        assert path, 'Set EXTERNAL_PATH to the exact folder (or zip) of the second dataset'
        if source == 'drive':
            from google.colab import drive; drive.mount('/content/drive'); return path
        if source == 'zip':
            out = os.path.join(WORK_DIR, 'data_' + tag); os.makedirs(out, exist_ok=True)
            with zipfile.ZipFile(path) as z: z.extractall(out)
            return out
        return path
    EXT_ROOT = get_root(EXTERNAL_SOURCE, EXTERNAL_KAGGLE_ID, EXTERNAL_PATH, 'external')
    assert os.path.isdir(EXT_ROOT), f'External folder not found: {EXT_ROOT}'
    assert os.path.abspath(EXT_ROOT) != os.path.abspath(DATA_ROOT), 'External data must be a different dataset from the training data'

    # scan with the external folder names, then restore the training ones
    _p, _n = POS_FOLDER, NEG_FOLDER
    POS_FOLDER, NEG_FOLDER = EXTERNAL_POS_FOLDER, EXTERNAL_NEG_FOLDER
    ext_items = scan(EXT_ROOT)
    POS_FOLDER, NEG_FOLDER = _p, _n
    assert len(ext_items) > 0, 'No labelled images found in the external dataset. Set EXTERNAL_POS_FOLDER / EXTERNAL_NEG_FOLDER.'
    n_found = len(ext_items)

    seen_hashes = {md5(it[0]) for it in train_items} | {md5(it[0]) for it in val_items}
    uniq, hashes, n_leak = [], set(), 0
    for it in ext_items:
        h = md5(it[0])
        if h in seen_hashes: n_leak += 1; continue          # exact copy of a training/validation image
        if h in hashes: continue                             # duplicate inside the external set
        hashes.add(h); uniq.append(it)
    ext_items = uniq
    print(f'External images found: {n_found} | copies of training/val images removed: {n_leak} | '
          f'other duplicates removed: {n_found - n_leak - len(ext_items)} | used: {len(ext_items)}')
    summary('ext', ext_items)
    assert len({it[1] for it in ext_items}) == 2, 'External set has only one class'

    ext_dl = DataLoader(XrayDS(ext_items, eval_tf), BATCH_SIZE, shuffle=False, num_workers=2)
    model.load_state_dict(torch.load(MODEL_FILE, map_location=device))
    _, ep, el = evaluate(ext_dl)
    epred = (ep >= THRESH).astype(int)
    etn, efp, efn, etp = [int(v) for v in confusion_matrix(el, epred, labels=[0, 1]).ravel()]

    def boot_ci(y, p, thr, n=1000, seed=0):
        rng = np.random.default_rng(seed); idx = np.arange(len(y)); aucs, ses, sps = [], [], []
        for _ in range(n):
            sm = rng.choice(idx, len(idx), replace=True); ys, ps = y[sm], p[sm]
            if len(set(ys)) < 2: continue
            pr = ps >= thr
            tp = ((pr == 1) & (ys == 1)).sum(); fn = ((pr == 0) & (ys == 1)).sum()
            tn = ((pr == 0) & (ys == 0)).sum(); fp = ((pr == 1) & (ys == 0)).sum()
            aucs.append(roc_auc_score(ys, ps)); ses.append(tp / max(tp + fn, 1)); sps.append(tn / max(tn + fp, 1))
        q = lambda a: [round(float(np.percentile(a, 2.5)), 4), round(float(np.percentile(a, 97.5)), 4)]
        return q(aucs), q(ses), q(sps)

    e_auc_ci, e_se_ci, e_sp_ci = boot_ci(el, ep, THRESH)
    i_auc_ci, i_se_ci, i_sp_ci = boot_ci(tl, tp, THRESH)
    d_ext = maha(embed(ext_dl))
    RESULTS['internal_ci'] = {'auc': i_auc_ci, 'sensitivity': i_se_ci, 'specificity': i_sp_ci}
    RESULTS['external'] = {
        'source': EXTERNAL_KAGGLE_ID or EXTERNAL_PATH, 'n_images': len(ext_items), 'copies_removed': n_leak,
        'auc': round(float(roc_auc_score(el, ep)), 4), 'auc_ci95': e_auc_ci,
        'sensitivity': round(etp / max(etp + efn, 1), 4), 'sensitivity_ci95': e_se_ci,
        'specificity': round(etn / max(etn + efp, 1), 4), 'specificity_ci95': e_sp_ci,
        'accuracy': round((etp + etn) / len(el), 4), 'confusion': {'TN': etn, 'FP': efp, 'FN': efn, 'TP': etp},
        'threshold_used': round(THRESH, 4), 'flagged_unsupported': round(float((d_ext > OOD_THRESH).mean()), 4)}
    print('\nEXTERNAL TEST RESULTS (model trained on the first dataset only)')
    print(f"{'':14s}{'internal test':>16s}{'external test':>16s}")
    print(f"{'AUC':14s}{RESULTS['auc']:>16.4f}{RESULTS['external']['auc']:>16.4f}   95% CI external {e_auc_ci}")
    print(f"{'sensitivity':14s}{RESULTS['sensitivity']:>16.4f}{RESULTS['external']['sensitivity']:>16.4f}   95% CI external {e_se_ci}")
    print(f"{'specificity':14s}{RESULTS['specificity']:>16.4f}{RESULTS['external']['specificity']:>16.4f}   95% CI external {e_sp_ci}")
    print('external confusion', RESULTS['external']['confusion'], '| flagged as unsupported:', f"{RESULTS['external']['flagged_unsupported']:.1%}")
    f2, t2, _ = roc_curve(el, ep)
    plt.plot(f2, t2, label='external'); plt.plot(*roc_curve(tl, tp)[:2], label='internal test'); plt.plot([0, 1], [0, 1], '--')
    plt.xlabel('False positive rate'); plt.ylabel('Sensitivity'); plt.legend(); plt.title('ROC: internal vs external test'); plt.show()

## 11. Save everything

In [ ]:
RESULT_FILE = f'results_{REGION}.json'
with open(RESULT_FILE, 'w') as fh: json.dump(RESULTS, fh, indent=2)
print('Saved:', MODEL_FILE, ',', RESULT_FILE, 'and', OOD_FILE)

if SAVE_TO_DRIVE and IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    out = '/content/drive/MyDrive/fracture_models'; os.makedirs(out, exist_ok=True)
    shutil.copy(MODEL_FILE, out); shutil.copy(RESULT_FILE, out); shutil.copy(OOD_FILE, out)
    csv = os.path.join(out, 'results_all_regions.csv'); new = not os.path.exists(csv)
    with open(csv, 'a') as fh:
        if new: fh.write('region,auc,sensitivity,specificity,accuracy,n_train,n_val,n_test\n')
        fh.write(f"{REGION},{RESULTS['auc']},{RESULTS['sensitivity']},{RESULTS['specificity']},{RESULTS['accuracy']},{RESULTS['n_train']},{RESULTS['n_val']},{RESULTS['n_test']}\n")
    print('Copied to Google Drive:', out)

if IN_COLAB:
    from google.colab import files
    files.download(MODEL_FILE); files.download(RESULT_FILE); files.download(OOD_FILE)
elif IN_KAGGLE:
    print('On Kaggle: the 3 files (model, results, ood) are in /kaggle/working. After the run finishes, open the notebook Output tab (or Save Version > Save & Run All) and download them there.')
else:
    print('Files saved in the current folder.')

## 12. Check one of your own images with this region's model

Colab: a Choose Files button appears. Kaggle: first set `IMAGE_PATH = '/kaggle/input/.../image.jpg'` in a cell. Use a clean exported X-ray (not a photo of a screen) of **this body part**. Learning tool only.

In [ ]:
if IN_COLAB:
    from google.colab import files as _f
    names = list(_f.upload().keys())
else:
    names = [IMAGE_PATH] if globals().get('IMAGE_PATH') else []   # Kaggle: set IMAGE_PATH = '/kaggle/input/<your-dataset>/xray.jpg' above, then re-run
    if not names: print('Set IMAGE_PATH to an image file path, then run this cell again.')
for name in names:
    img = eval_tf(dip_preprocess(Image.open(name).convert('RGB'), PREPROCESS, IMG_SIZE))
    plt.figure(figsize=(5, 5)); p = show_cam(img)
    plt.title(('POSSIBLE FRACTURE' if p >= THRESH else 'No fracture detected') + f'\nP(fracture)={p:.2f}'); plt.show()
    print('Learning tool only. Ask a doctor for a real diagnosis.')

## For the next body part

1. Change `REGION`, `SOURCE` and `DATA_PATH` (or `KAGGLE_ID`) in cell 1.
2. **Runtime > Run all**.
3. Keep all 3 files for every region: `model_<region>.pt`, `results_<region>.json`, `ood_<region>.npz`. The app will load them all later.

**Report tip:** put each region's `results_<region>.json` numbers in one table, and state the limits (single source dataset, exact-duplicate check only, no external validation).